<p> <center><img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:200px"> </p>

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Analyse et prédiction de la variabilité de la production solaire à partir de données ouvertes de la région PACA </h1></center>
<center><h2> Collecte des données de production d'énergie solaire </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">


La variable dont on souhaite étudier la variabilité est le **Taux de CHarge de la production solaire** (TCH solaire). On peut trouver cette variable dans les données mises à disposition par RTE dans un dataset appelé **éCO2mix**.

Si cette base de données nous fourni une variable nous permettant de calculer notre variable cible, elle ne comprend **aucune donnée météo ou astronomique** permettant d'expliquer la production d'énergie solaire : nous savons donc dès le départ que nous devrons **compléter** cette base par d'autres données.

Or la base éCO2mix se présente sous forme d'une série temporelle : on pourra donc partir de la **variable temporelle** pour aggréger de nouvelles données par la suite.

Concrètement, à cette étape de la collecte, il nous faut :

* **Récupérer** les données d'éCO2mix.
* Ne **conserver** que les observations où notre **base de calcul** (TCH solaire) pour la variable cible  est présente.
* **Abandonner** les données ne concernant pas l'énergie solaire (la base éCO2mix contient des données concernant d'autres types d'énergies).
* Vérifier le **fuseau horaire** de la variable temporelle et le convertir en **UTC** si besoin, de manière à pouvoir aggréger facilement des données d'autres datasets.


# I - Acquisition des données brutes

On peut trouver les données brutes sur le site Internet de **RTE** (https://www.rte-france.com/donnees-publications/eco2mix-donnees-temps-reel/telecharger-indicateurs).

Sur ce site, on dispose de jeux de données concernant la **région PACA** pour les années **2013 à 2024** incluses, sous forme d'archives comprimées, à raison d'**une archive par année**.

On a préalablement téléchargé ces données et on les a placées dans un répertoire de travail dont on stocke le chemin dans une variable `chemin_datasets`.

In [1]:
# Répertoire des données d'éCO2mix téléchargées
input_datasets = '../../data/local_data/02_Collecte_datasets/01_Production/input/'

# Répertoire où seront stockés les résultats
output_datasets = '../../data/local_data/02_Collecte_datasets/01_Production/output/'

On importe les librairies dont on va avoir besoin pour manipuler nos jeux de données :

In [2]:
# Importer des librairies usuelles pour traiter des datasets
import pandas as pd
import numpy as np

# Importer une librairie pour traiter des fichiers zip
from zipfile import ZipFile

Pour consulter nos datasets, nous devons préalablement lister les noms des archives puis les décompresser.

In [3]:
# Construire une liste des noms des archives à traiter, sans l'extension

## Base des noms des fichiers de production
file_base_name = input_datasets + 'eCO2mix_RTE_PACA_Annuel-Definitif_'
last_file_base_name = input_datasets + 'eCO2mix_RTE_PACA_En-cours-Consolide' # concerne l'année 2024

## Construction de la liste des bases de noms de fichers
annees_dispos = [a for a in range(2013, 2024)]
file_list = []
for annee in annees_dispos:
    file_list.append(file_base_name + str(annee))

file_list.append(last_file_base_name)

print(file_list)


# Décompresser les archives

## Parcourir la liste des archives
for f in file_list :
    # Extraire chaque archive
    with ZipFile(f + '.zip', 'r') as zip_file :
        zip_file.printdir()
        zip_file.extractall(input_datasets)


# II - Première exploration des données éCO2mix

La décompression des archives nous a permis d'obtenir des datasets dans un format *.xls*. Cependant, après examen, ces datasets se révèlent être en fait au format *.csv*, encodés en *latin-1*.

On va maintenant commencer à explorer nos jeux de données et en particulier vérifier la **présence** de notre **base de calcul** (TCH Solaire) pour la **variable cible** (variabilité de la production d'énergie solaire).

In [4]:
# Observer les premières lignes du dataset le plus ancien (correspond à l'année 2013) :
print("\nJeu de données 2013 :")

## Lire le jeu de données
df = pd.read_csv(file_list[0] + '.xls', sep='\t', encoding='latin_1')

## Afficher les premières lignes
display(df.head())

## Afficher la taille du dataset
print(f"Nombre de colonnes du jeu de données 2013 : {df.shape[1]}.")


# Observer les premières lignes du dataset le plus récent (correspond à l'année 2024) :
print("\nJeu de données 2024 :")

## Lire le jeu de données
### Malgré son extension, le fichier xls est en fait un fichier csv encodé en 'iso_8859-1' ce qui correspond au 'latin_1'
df = pd.read_csv(file_list[-1] + '.xls', sep='\t', encoding='latin_1')

## Afficher les premières lignes
display(df.head())

## Afficher la taille du dataset
print(f"Nombre de colonnes du jeu de données 2024 : {df.shape[1]}.")


En comparant quelques lignes des **premier et dernier datasets**, on remarque que **le nombre de colonnes est différent** (14 colonnes en 2013 contre 30 en 2024).

Les *variables disponibles* ne sont *pas les mêmes d'une année à l'autre*.

Par ailleurs, **l'année 2013 ne comporte pas** la variable `TCH Solaire (%)` et ne nous permet pas de calculer **notre variable cible**.


# III - Sélection des datasets contenant la variable TCH Solaire

Pour notre projet, nous avons besoin de données où la variable `TCH Solaire (%)` est présente.

Nous allons chercher les datasets disponibles qui la comporte.

In [5]:
# Chercher la présence de la variable 'TCH Solaire (%)'

year = 2013 # Sert  à afficher l'année en cours de traitement
var_base_cible = 'TCH Solaire (%)' # Nom de la variable nécessaire au calcul de la variable cible

# On parcours les datasets
for f in file_list :

    # Lire le jeu de données
    # Malgré son extension, le fichier xls est en fait un fichier csv encodé en 'iso_8859-1' ce qui correspond au 'latin_1'
    df = pd.read_csv(f + '.xls', sep='\t', encoding='latin_1')

    # On affiche des informations basiques sur le dataset qu'on vient d'extraire
    print(year, ":") # L'année correspondant au dataset en cours de traitement
    print(f"\tTaille du dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes.") # les dimensions du dataset
    print("\tBase de la cible présente :", (var_base_cible in df.columns)) # la variable est-elle présente dans le dataset

    year += 1


Seules les années **2020 à 2024** incluses comprennent la variable `TCH Solaire (%)` : les datasets que nous décidons de conserver sont ceux correspondants à ces années.

# IV - Variables communes aux datasets retenus

Nos souhaitons n'avoir qu'un **unique dataset** comprenant l'ensemble des données de **2020 à 2024** inclus.

Or, nous avons vu que les datasets de différentes années peuvent avoir des **variables différentes** : nous avons besoin de lister les **variables communes** à tous les datasets.

Pour cela, nous allons **lister les variables** présentes dans les datasets, puis prendre les variables d'un des datasets et regarder si elles sont présentes dans les autres datasets. On ne conservera que celles **présentes dans tous les datasets**.

In [6]:
# Lire les datasets dans des DataFrames
## Lire fichier source du dataset de 2020
df_2020 = pd.read_csv(input_datasets + 'eCO2mix_RTE_PACA_Annuel-Definitif_2020.xls', sep='\t', encoding='latin_1')
## Lire fichier source du dataset de 2021
df_2021 = pd.read_csv(input_datasets + 'eCO2mix_RTE_PACA_Annuel-Definitif_2021.xls', sep='\t', encoding='latin_1')
## Lire fichier source du dataset de 2022
df_2022 = pd.read_csv(input_datasets + 'eCO2mix_RTE_PACA_Annuel-Definitif_2022.xls', sep='\t', encoding='latin_1')
## Lire fichier source du dataset de 2023
df_2023 = pd.read_csv(input_datasets + 'eCO2mix_RTE_PACA_Annuel-Definitif_2023.xls', sep='\t', encoding='latin_1')
## Lire fichier source du dataset de 2024
df_2024 = pd.read_csv(input_datasets + 'eCO2mix_RTE_PACA_En-cours-Consolide.xls', sep='\t', encoding='latin_1')


# Stocker les DataFrames dans la variable dataset_list
dataset_list = [df_2020, df_2021, df_2022, df_2023, df_2024]


# Etablir la liste de toutes les colonnes des datasets retenus
liste_colonnes = []

## Déclarer puis initialiser la liste des colonnes des datasets en parcourant ces derniers
for df in dataset_list :
    liste_colonnes.append(df.columns)


# Etablir la liste des colonnes présentes dans tous les datasets retenus
variables_retenues = []

## On part des colonnes du premier dataset
for col in liste_colonnes[0]:
    ### La variable col_in_all vaut True si la colonne est présente dans tous les datasets, False sinon
    col_in_all = True # Par défaut on considère que la colonne actuelle est dans les autres datasets

    ### On parcourt les colonnes des autres datasets
    for cols in liste_colonnes[1:] :
        if col not in cols : # Si la colonne courante n'est pas dans le dataset courant
            col_in_all = False # On met à jour col_in_all

    ### Si la colonne est bien présente dans tous les datasets, on l'ajoute à la variables_retenues
    if col_in_all:
        variables_retenues.append(col)


# On affiche la taille et le contenu de la liste finale
print("\nTaille de la liste de variables retenues :", len(variables_retenues))
print("Variables retenues :", variables_retenues)

Nous avons maintenant la **liste des variables communes aux datasets comprenant la variable permettant de calculer la variable cible**.

On remarque cependant dans cette liste les variables **ne concernent pas** uniquement la **production d'énergie solaire**, mais également la production d'énergie thermique, nucléaire, éolienne, ...

Ces variables risquent de perturber notre futur modèle qui se concentre uniquement sur l'énergie solaire : nous décidons de les **abandonner**.

In [7]:
# On supprime de la liste les colonnes dont on sait qu'elles ne concernent pas l'énergie solaire
variables_retenues.remove('Thermique')
variables_retenues.remove('Nucléaire')
variables_retenues.remove('Eolien')
variables_retenues.remove('Hydraulique')
variables_retenues.remove('Pompage')
variables_retenues.remove('Bioénergies')
variables_retenues.remove('TCO Thermique (%)')
variables_retenues.remove('TCH Thermique (%)')
variables_retenues.remove('TCO Nucléaire (%)')
variables_retenues.remove('TCH Nucléaire (%)')
variables_retenues.remove('TCO Eolien (%)')
variables_retenues.remove('TCH Eolien (%)')
variables_retenues.remove('TCO Hydraulique (%)')
variables_retenues.remove('TCH Hydraulique (%)')
variables_retenues.remove('TCO Bioénergies (%)')
variables_retenues.remove('TCH Bioénergies (%)')

# On affiche les informations et le contenu de la liste finale
print("Taille de la liste de variables retenues :", len(variables_retenues))
print("Variables retenues :", variables_retenues)

# V - Concaténation des datasets

Maintenant que nous disposons d'une liste de variables communes qui semble pertinente à ce stade, nous pouvons **regrouper les différents datasets en un seul** dataset `df_rte` pour faciliter la suite de l'étude.

In [8]:
# Concaténation des datasets
# On affiche le nombre de lignes total qu'on est supposé obtenir
print("Nombre total de lignes avant concaténation :",
      df_2020.shape[0] + df_2021.shape[0] + df_2022.shape[0] + df_2023.shape[0] + df_2024.shape[0])

# On concatène sur les colonnes retenues
df_rte = pd.concat([df_2020[variables_retenues],
                       df_2021[variables_retenues],
                       df_2022[variables_retenues],
                       df_2023[variables_retenues],
                       df_2024[variables_retenues]],
                      axis = 0)

# On affiche le nombre de lignes obtenu
print(f"Le dataset concaténé obtenu présente {df_rte.shape[0]} lignes et {df_rte.shape[1]} colonnes.") # OK

# VI - Elimination des observations ne comprenant pas la variable TCH Solaire

Faisons maintenant un premier examen de notre nouveau jeu de données.

In [9]:
display(df_rte.head(10))
display(df_rte.describe())
df_rte.info()

RTE indique dans les spécifications du jeu de données **éCO2mix** que ce dernier a une résolution temporelle de 15 minutes. On remarque cependant rapidement que les données d'une ligne sur deux des datasets ne présente ni donnée de production ni donnée de consommation : la **résolution temporelle** semble en réalité être de **30 minutes**.

En particulier *une ligne sur deux ne contient pas la variable* `TCH Solaire (%)`.

Or nous avons besoin de cette variable pour la construction de nos futurs modèles : nous **supprimons** donc les observations qui ne la comprennent pas.

In [10]:
# On supprime les lignes où notre variable TCH Solaire est manquante
var_base_cible = 'TCH Solaire (%)'
df_rte = df_rte.dropna(subset=[var_base_cible])

On vérifie s'il reste des valeurs manquantes :

In [11]:
print(df_rte.isna().sum())

Il ne reste **aucune valeur manquante** après l'élimination des observations ne comprenant pas la variable `TCH Solaire (%)`.

# VII - Fuseau horaire

Les **spécifications du jeu de données** éCO2mix décrivant les variables présentes dans ce jeu de données **ne précisent pas le fuseau horaire** auquel les mesures ont été réalisées.

Comme RTE est une entreprise Française et qu'on s'intéresse à la région PACA, on fait l'hypothèse de l'utilisation du **fuseau horaire de France métropolitaine**.

Afin de vérifier cette hypothèse, on va regarder si on observe un passage heure d'été - heure d'hiver grâce aux données de production d'énergie solaire.

Comment se comporte la production entre deux dates où on sait qu'il y a changement d'horaire pour ce fuseau ? **Observe-t-on un décalage horaire dans la production** entre le jour avant et le jour suivant le changement d'heure ?

 - Si oui : le fuseau horaire est bien celui de France métropolitaine ;
 - Si non : le fuseau horaire est peut être le fuseau UTC (dans ce cas un courriel de demande de renseignements aurès de RTE pourrait permettre d'en être sûr).

Comme les datasets n'avaient initialement pas tous le même nombre de variables, on ne sait pas s'ils ont tous été construits avec le même fuseau horaire donc *on vérifie pour les créneaux de changement d'heure de 2020 à 2024*.

Les changements d'heure en France métropolitaine ont eu lieu :
 - En 2020 : entre les 28-29 mars et 24-25 octobre ;
 - En 2021 : entre les 27-28 mars et 30-31 octobre ;
 - En 2022 : entre les 26-27 mars et 29-30 octobre ;
 - En 2023 : entre les 25-26 mars et 28-29 octobre ;
 - En 2024 : entre les 30-31 mars et 26-27 octobre.

In [12]:
# On importe une librairie de Datavizualisation
import matplotlib.pyplot as plt
%matplotlib inline

In [13]:
# Extraire les variables pertinentes (date, heure et TCH Solaire)
df = df_rte[['Date', 'Heures', var_base_cible]]

# Lister les jours avant et après les changements d'heure pour le fuseau France métropolitaine
## Jours avant les changements d'heure d'été
ete_av = ['2020-03-28', '2021-03-27', '2022-03-26', '2023-03-25', '2024-03-30']
## Jours après les changements d'heure d'été
ete_ap = ['2020-03-29', '2021-03-28', '2022-03-27', '2023-03-26', '2024-03-31']
## Jours avant les changements d'heure d'hiver
hiv_av = ['2020-10-24', '2021-10-30', '2022-10-29', '2023-10-28', '2024-10-26']
## Jours après les changements d'heure d'hiver
hiv_ap = ['2020-10-25', '2021-10-31', '2022-10-30', '2023-10-29', '2024-10-27']

# Redimensionner la figure (on va afficher une ligne par année et une colonne par changement d'heure)
plt.figure(figsize=(15,30))

# Pour chaque année
for i in range(5):

  # On affiche la production d'énergie solaire avant et après le passage à l'heure d'été
  plt.subplot(5, 2, i*2+1)
  plt.plot(df[df['Date'] == ete_av[i]].Heures, df[df['Date'] == ete_av[i]][var_base_cible], label='Avant')
  plt.plot(df[df['Date'] == ete_ap[i]].Heures, df[df['Date'] == ete_ap[i]][var_base_cible], label='Après')
  plt.xlabel('Heure')
  plt.ylabel(var_base_cible)
  # On nettoie les ticks sur l'axe des x car sinon on ne peut rien lire
  # On n'affiche que les heures
  plt.xticks([2*k for k in range(0, 25)],
            [str(k) for k in range(25)],
            rotation=30)
  plt.title("Passage à l'heure d'été "+str(2020+i))
  plt.legend()

  # On affiche la production d'énergie solaire avant et après le passage à l'heure d'hiver
  plt.subplot(5, 2, i*2+2)
  plt.plot(df[df['Date'] == hiv_av[i]].Heures, df[df['Date'] == hiv_av[i]][var_base_cible], label='Avant')
  plt.plot(df[df['Date'] == hiv_ap[i]].Heures, df[df['Date'] == hiv_ap[i]][var_base_cible], label='Après');
  plt.xlabel('Heure')
  plt.ylabel(var_base_cible)
  # On nettoie les ticks sur l'axe des x car sinon on ne peut rien lire
  # On n'affiche que les heures
  plt.xticks([2*k for k in range(0, 25)],
            [str(k) for k in range(25)],
            rotation=30)
  plt.title("Passage à l'heure d'hiver "+str(2020+i))
  plt.legend()

plt.show()


L'hypothèse du *fuseau horaire de France métropolitaine* se confirme puisqu'**on observe systématiquement un décalage horaire** de la production d'électricité par panneaux solaires le jour suivant un changement d'heure.

# VIII - Conversion au fuseau horaire UTC

Nous venons de voir que nos données de production étaient données en fonction du fuseau horaire de France métropolitaine.

Il y a deux raisons pour lesquelles nous devrions **changer de fuseau horaire** :    
 - Premièrement le fuseau horaire actuel introduit une modification du *décalage entre l'heure solaire et l'heure légale* deux fois par an : cela **risque de perturber** notre modélisation ;

 - Deuxièmement nous allons devoir *aggréger de nouvelles variables explicatives* à notre jeu de données pour pouvoir expliquer notre variable cible. Comme nous interrogerons des **bases internationnales**, il nous faut convertir nos dates et heures en UTC.

On va créer une nouvelle variable `datetime_utc` qui contiendra les mêmes informations que les variables `Date` et `Heures`, mais en heures UTC, de manière à pouvoir aggréger plus facilement d'autres données par la suite, sur la base de cette variable.

In [14]:
# Aggréger la date et l'heure FR dans une nouvelle colonne
df_rte['datetime_fr'] = pd.to_datetime(df_rte['Date'] + ' ' + df_rte['Heures'], format="%Y-%m-%d %H:%M")

# Indiquer que l'heure est celle de Paris
df_rte['datetime_fr'] = df_rte['datetime_fr'].dt.tz_localize("Europe/Paris",
                                                                 nonexistent='shift_forward',
                                                                 ambiguous=True)
# Convertir en UTC
df_rte['datetime_utc'] = df_rte['datetime_fr'].dt.tz_convert("UTC")


# IX - Passage à l'heure d'été

Observons un passage à l'heure d'été pour voir comment s'est déroulée la conversion.

Lors d'un passage à l'heure d'été, une heure est "perdue" pendant la nuit : par exemple lorsqu'il devait être 2h du matin sans changement d'heure, il est 3h du matin avec changement d'heure.

Il devrait y avoir deux créneaux horaires manquants en heure française pour chaque passage à l'heure d'été.

In [15]:
# On observe la conversion à une date où on sait qu'il y a eu un passage à l'heure d'été
display(df_rte[df_rte['Date'] == '2024-03-31'])
print(df_rte[df_rte['Date'] == '2024-03-31'].shape)

On constate que le jeu de données ne contient **pas de créneau manquant** à l'heure "perdue", mais que les données de l'heure inexistante sont une copie des deux observations suivantes (déduit par l'observation des variables `Consommation`et `Ech. physique`).

Comme le changement d'heure à lieu en pleine nuit lorsque les panneaux solaires ne produisent pas d'énergie, l'*impact de ces données synthétiques semble minime* si on reste dans le fuseau horaire de départ.

Cependant comme on souhaite changer de fuseau, cela introduit des **doublons de créneaux horaires UTC** dans nos données : il nous faut les supprimer.

In [16]:
# Voir les duplicatas au moment du passage à l'heure d'été
display(df_rte[df_rte.duplicated(subset=['datetime_utc'])]) # Tous les dulicatas correspondent à des passages à l'heure d'été
print("Nombre de duplicata :", df_rte.duplicated(subset=['datetime_utc']).sum())

In [17]:
# On supprime les duplicatas
df_rte = df_rte.drop_duplicates(subset=['datetime_utc'])
print("Nombre de duplicata restants :", df_rte.duplicated(subset=['datetime_utc']).sum())

# X - Passage à l'heure d'hiver

Observons maintenant un passage à l'heure d'hiver pour voir comment s'est déroulée la conversion.

Lors d'un passage à l'heure d'hiver, une heure est "gagnée" pendant la nuit : par exemple lorsqu'il devait être 3h du matin sans changement d'heure, il est 2h du matin avec changement d'heure. Cela va écraser les données de deux créneaux horaires à chaque passage à l'heure d'hiver pour le fuseau de France métropolitaine.

Il devrait donc y avoir deux créneaux horaires manquants en heure UTC pour chaque passage à l'heure d'hiver.


In [18]:
# On observe la conversion à une date où on sait qu'il y a eu un passage à l'heure d'hiver
display(df_rte[df_rte['Date'] == '2024-10-27'])
print(df_rte[df_rte['Date'] == '2024-10-27'].shape)

On observe effectivement des créneaux manquants dans la variables `datatime_utc`.

Voyons s'il existe d'autres créneaux manquants (par exemple s'il y a eu un arret de service de collecte des données) avant de traiter ceux dûs aux changements d'heure d'hiver.

In [19]:
# Recherche des créneaux manquants

# On commence par faire la liste des créneaux qui devraient être présents dans le jeu de données
# Un créneau toute les 30 minutes entre le premier et le dernier créneau horaire du dataset
totalite_creneaux = pd.date_range(start=df_rte['datetime_utc'].iloc[0], end=df_rte['datetime_utc'].iloc[-1], freq='30min')

# Créneaux manquants
print("Créneaux manquants :")
creneaux_manquants = totalite_creneaux.difference(df_rte['datetime_utc'])
print(creneaux_manquants) # Les seuls créneaux manquants correspondent au passage à l'heure d'hiver


Les **seuls créneaux manquants** sont ceux dûs aux **passages à l'heure d'hiver**.

On décide de traiter ces créneaux manquants *de la même manière* que ceux de l'heure inexistante au moment du passage à l'heure d'été, c'est à dire qu'on **copie les données de l'heure suivante**.

Comme vu précédemment, l'**impact** de ces données synthétique pour notre problématique de production d'énergie solaire est **minime** vu que les changement d'heure ont lieu à un moment où *les panneaux solaires ne produisent pas* d'énergie.

In [20]:
print(df_rte.shape)
df_rte2 = pd.DataFrame(totalite_creneaux, columns=['datetime_utc'])
df_rte = df_rte2.merge(df_rte, how='left', on='datetime_utc')
print(df_rte.shape)
print(f"\n{df_rte.isna().sum()}")
df_rte.sort_values(by='datetime_utc', inplace=True)

for creneau in creneaux_manquants:
    df_rte.loc[df_rte['datetime_utc']== creneau, 'Périmètre':'datetime_fr'] = df_rte.loc[df_rte['datetime_utc'] == creneau + pd.Timedelta(hours=1), 'Périmètre':'datetime_fr'].values

print(f"\n{df_rte.isna().sum()}")
display(df_rte[df_rte['Date'] == '2024-10-27'])

On abandonne maintenant les colonnes de dates inutiles pour ne conserver que le **fuseau horaire UTC** :

In [21]:
# On supprime les colonnes 'Date', 'Heures' et 'datetime_fr'
df_rte = df_rte.drop(['Date', 'Heures', 'datetime_fr'], axis=1)

Enfin, on réinitialise l'index :

In [22]:
# On réinitialise l'index
df_rte.reset_index(drop=True, inplace=True)

display(df_rte.head())
print(df_rte.shape)

# XI - Enregistrement des données obtenues de RTE

Nous avons terminé la collecte des données contenant notre variable `TCH Solaire (%)` depuis le jeu de données éCO2mix gracieusement fourni par RTE et l'avons préparé pour pouvoir y aggréger de nouvelles données d'**autres sources**.

Nous enregistrons notre jeu de données actuel pour clore la première partie de notre travail de collecte.

In [23]:
# On enregistre une seconde version de ce dataset production avant ajout d'autres variables
df_rte.to_csv(output_datasets + "production_2020_2024.csv", index=False)

In [24]:
# Exemple de la manière dont charger ce dataset production final :
df = pd.read_csv(output_datasets + "production_2020_2024.csv", parse_dates=['datetime_utc'])
display(df.head())
df.dtypes